In [1]:
import os
import json
import mlflow
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import  roc_auc_score

/workspaces/MLOps-Movie-Hit-Flop-Predictor/venv/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
def load_and_explore_data(file_path):
    """Load data and return basic info"""
    df = pd.read_csv(file_path)
    
    # print(f"Dataset shape: {df.shape}")
    # print(f"\nColumns: {df.columns.tolist()}")
    # print(f"\nMissing values:\n{df.isnull().sum()}")
    # print(f"\nFirst few rows:\n{df.head()}")
    
    return df

In [3]:
def clean_data(df):
    """Clean and prepare the dataset"""
    print(f"Before cleaning: {len(df)} rows")
    
    # Remove duplicates
    df = df.drop_duplicates()
    print(f"After duplicate removal: {len(df)} rows")
    
    # Remove zero runtime and budget (data errors)
    df = df[df['runtime'] > 0]
    df = df[df['budget'] > 0]
    print(f"After cleaning: {len(df)} rows")
    
    # Convert release_date
    df['release_date'] = pd.to_datetime(df['release_date'])
    
    return df.copy()

In [4]:
def create_target_variable(df):
    """Create ROI-based hit/flop target"""
    df['roi'] = df['revenue'] / (df['budget'] + 1)
    roi_threshold = df['roi'].quantile(0.7)
    df['is_hit'] = (df['roi'] >= roi_threshold).astype(int)
    
    print(f"ROI threshold: {roi_threshold:.2f}")
    print(f"Hit distribution:\n{df['is_hit'].value_counts()}")
    print(f"Hit percentage: {df['is_hit'].mean()*100:.1f}%")
    
    return df

print("Data processing functions defined!")

Data processing functions defined!


In [5]:
def extract_genres(genres_str):
    """Extract genre count and main genre from JSON string"""
    try:
        genres = json.loads(genres_str.replace("'", '"'))
        return len(genres), genres[0]['name'] if genres else 'Unknown'
    except:
        return 0, 'Unknown'

In [6]:
def engineer_features(df):
    """Create all engineered features"""
    # Time-based features
    df['release_year'] = df['release_date'].dt.year
    
    # Budget categories
    df['budget_category'] = pd.cut(df['budget'], 
                                   bins=[0, 1000000, 50000000, 150000000, float('inf')], 
                                   labels=['Ultra_Low', 'Low', 'Medium', 'High'])
    
    # Genre features
    df[['genre_count', 'main_genre']] = df['genres'].apply(
        lambda x: pd.Series(extract_genres(x))
    )
    
    # Language feature
    df['is_english'] = (df['original_language'] == 'en').astype(int)
    
    print("Features engineered:")
    print(f"Release year range: {df['release_year'].min()}-{df['release_year'].max()}")
    print(f"Budget categories: {df['budget_category'].value_counts()}")
    print(f"Genre count range: {df['genre_count'].min()}-{df['genre_count'].max()}")
    print(f"English movies: {df['is_english'].mean()*100:.1f}%")
    
    return df

In [7]:
def prepare_features(df):
    """Select and prepare features for modeling"""
    numeric_features = ['budget', 'runtime', 'vote_average', 
                       'vote_count', 'popularity', 'genre_count', 'release_year']
    categorical_features = ['budget_category', 'main_genre', 'is_english']
    
    # Handle missing values
    df['release_year'].fillna(df['release_year'].median(), inplace=True)
    df['budget_category'].fillna('Ultra_Low', inplace=True)
    
    # Prepare feature matrix
    X = df[numeric_features + categorical_features].copy()
    
    # Encode categorical variables
    le_budget = LabelEncoder()
    le_genre = LabelEncoder()
    
    X['budget_category'] = le_budget.fit_transform(X['budget_category'])
    X['main_genre'] = le_genre.fit_transform(X['main_genre'])
    
    y = df['is_hit']
    
    print(f"Final feature matrix: {X.shape}")
    print(f"Features: {X.columns.tolist()}")
    
    return X, y, le_budget, le_genre

print("Feature engineering functions defined!")

Feature engineering functions defined!


In [8]:
def setup_mlflow():
    """Initialize MLflow tracking"""
    mlflow_dir = "../mlflow_data"
    os.makedirs(mlflow_dir, exist_ok=True)
    
    mlflow.set_tracking_uri(f"sqlite:///{mlflow_dir}/mlflow.db")
    mlflow.set_experiment("movie-hit-prediction")
    os.environ['MLFLOW_DEFAULT_ARTIFACT_ROOT'] = f"{mlflow_dir}/artifacts"
    
    print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
    return mlflow_dir

In [9]:
def prepare_train_test_split(X, y):
    """Split data and handle class imbalance"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Apply SMOTE for class balancing
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f"Training set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    print(f"Original training: {np.bincount(y_train)}")
    print(f"Balanced training: {np.bincount(y_train_balanced)}")
    
    return X_test, y_test, X_train_balanced, y_train_balanced

In [10]:
def train_best_model(X_train_balanced, y_train_balanced, X_test, y_test):
    """Train and tune the best performing model (Random Forest) with balanced data"""
    
    # Hyperparameter tuning
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    }
    
    with mlflow.start_run(run_name="random_forest_tuned_final"):
        grid_search = GridSearchCV(
            RandomForestClassifier(random_state=42), 
            param_grid, cv=3, scoring='roc_auc'
        )
        grid_search.fit(X_train_balanced, y_train_balanced)  # Using balanced data
        
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        # Log to MLflow
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("auc", auc)
        mlflow.sklearn.log_model(best_model, "random_forest_model")
        
        # Register model
        model_uri = f"runs:/{mlflow.active_run().info.run_id}/random_forest_model"
        mlflow.register_model(model_uri, "movie_hit_predictor")
        
        print(f"Best params: {grid_search.best_params_}")
        print(f"Final Model - Accuracy: {accuracy:.3f}, AUC: {auc:.3f}")
        print(f"Model registered: {model_uri}")
        
        return best_model, accuracy, auc, model_uri

In [11]:
def run_movie_prediction_pipeline(data_path='../data/popular_movies.csv'):
    """
    Complete ML pipeline for movie hit prediction
    Returns: trained model, performance metrics, and artifacts
    """
    print("=== STARTING MOVIE HIT PREDICTION PIPELINE ===")
    
    # Step 1: Data Processing
    print("\n1. Loading and cleaning data...")
    df = load_and_explore_data(data_path)
    df_clean = clean_data(df)
    df_clean = create_target_variable(df_clean)
    
    # Step 2: Feature Engineering
    print("\n2. Engineering features...")
    df_clean = engineer_features(df_clean)
    X, y, le_budget, le_genre = prepare_features(df_clean)
    
    # Step 3: Model Training
    print("\n3. Setting up MLflow and training...")
    setup_mlflow()
    # Only get what we need for training
    X_test, y_test, X_train_balanced, y_train_balanced = prepare_train_test_split(X, y)
    
    # Use balanced data for training
    best_model, final_accuracy, final_auc, model_uri = train_best_model(X_train_balanced, y_train_balanced, X_test, y_test)
    
    # Step 4: Return results
    results = {
        'model': best_model,
        'accuracy': final_accuracy,
        'auc': final_auc,
        'model_uri': model_uri,
        'encoders': {'budget': le_budget, 'genre': le_genre},
        'test_data': (X_test, y_test)
    }
    
    print("\n=== PIPELINE COMPLETED SUCCESSFULLY ===")
    return results

print("Main pipeline function defined!")

Main pipeline function defined!


In [12]:
# Execute complete pipeline
if __name__ == "__main__":
    results = run_movie_prediction_pipeline()
    
    print(f"\nFinal Results:")
    print(f"Model Type: Random Forest")
    print(f"Accuracy: {results['accuracy']:.3f}")
    print(f"AUC: {results['auc']:.3f}")
    print(f"Model registered at: {results['model_uri']}")
    
    # Feature importance
    feature_names = ['budget', 'runtime', 'vote_average', 'vote_count', 
                    'popularity', 'genre_count', 'release_year', 
                    'budget_category', 'main_genre', 'is_english']
    importances = results['model'].feature_importances_
    
    print("\nTop 5 Important Features:")
    for name, importance in sorted(zip(feature_names, importances), 
                                  key=lambda x: x[1], reverse=True)[:5]:
        print(f"{name}: {importance:.3f}")

=== STARTING MOVIE HIT PREDICTION PIPELINE ===

1. Loading and cleaning data...
Before cleaning: 9999 rows
After duplicate removal: 9407 rows
After cleaning: 4995 rows
ROI threshold: 3.89
Hit distribution:
is_hit
0    3496
1    1499
Name: count, dtype: int64
Hit percentage: 30.0%

2. Engineering features...


Features engineered:
Release year range: 1920-2026
Budget categories: budget_category
Low          3421
Medium        998
Ultra_Low     359
High          217
Name: count, dtype: int64
Genre count range: 0-8
English movies: 84.9%
Final feature matrix: (4995, 10)
Features: ['budget', 'runtime', 'vote_average', 'vote_count', 'popularity', 'genre_count', 'release_year', 'budget_category', 'main_genre', 'is_english']

3. Setting up MLflow and training...
MLflow tracking URI: sqlite:///../mlflow_data/mlflow.db
Training set: (3996, 10)
Test set: (999, 10)
Original training: [2797 1199]
Balanced training: [2797 2797]


Registered model 'movie_hit_predictor' already exists. Creating a new version of this model...
2025/08/03 10:35:09 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: movie_hit_predictor, version 7


Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Final Model - Accuracy: 0.771, AUC: 0.820
Model registered: runs:/530f90e9275b4b36bd89f95ad6268c1e/random_forest_model

=== PIPELINE COMPLETED SUCCESSFULLY ===

Final Results:
Model Type: Random Forest
Accuracy: 0.771
AUC: 0.820
Model registered at: runs:/530f90e9275b4b36bd89f95ad6268c1e/random_forest_model

Top 5 Important Features:
vote_count: 0.203
popularity: 0.162
budget: 0.159
vote_average: 0.129
release_year: 0.122


Created version '7' of model 'movie_hit_predictor'.
